# Set up

In [1]:
!pip install pyreadstat basedosdados

import glob
import os
import shutil
import time
from pathlib import Path

import basedosdados as bd
import pandas as pd
import pyreadstat
from google.cloud import bigquery
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


# Funções

In [2]:
def to_partitions(
    data: pd.DataFrame, partition_columns: list[str], savepath: str
):
    """Salva o dataframe particionado em pastas hive (ex.: ano=2025/sigla_uf=SP/)."""

    if isinstance(data, (pd.core.frame.DataFrame)):
        savepath = Path(savepath)

        # combinações únicas entre as colunas de partição
        unique_combinations = (
            data[partition_columns]
            .drop_duplicates(subset=partition_columns)
            .to_dict(orient="records")
        )

        for filter_combination in unique_combinations:
            patitions_values = [
                f"{partition}={value}"
                for partition, value in filter_combination.items()
            ]

            # filtra os dados dessa combinação
            df_filter = data.loc[
                data[filter_combination.keys()]
                .isin(filter_combination.values())
                .all(axis=1),
                :,
            ]
            df_filter = df_filter.drop(columns=partition_columns)

            # cria a árvore de pastas
            filter_save_path = Path(savepath / "/".join(patitions_values))
            filter_save_path.mkdir(parents=True, exist_ok=True)
            file_filter_save_path = Path(filter_save_path) / "student.csv"

            # acrescenta os dados ao csv
            df_filter.to_csv(
                file_filter_save_path,
                index=False,
                mode="a",
                header=not file_filter_save_path.exists(),
            )
    else:
        raise BaseException("Data need to be a pandas DataFrame")


def contar_linhas(caminho):
    """Numero de linhas de dado num csv (desconta o cabecalho)."""
    with open(caminho, encoding="utf-8") as f:
        return sum(1 for _ in f) - 1

# 2025

In [6]:
BASE = "/content/drive/Shareddrives/Base dos Dados - Geral/Dados/Conjuntos/mundo_oecd_pisa/"
ARQUITETURA = (
    BASE + "extra/architecture/student.csv"
)  # suba a versao final (294 linhas) para o Colab
SAV = BASE + "input/2025/CY09_MS_STU_PUF.sav"
OUTPUT = BASE + "output2/year=2025"
PARTICAO = "country_id_iso_3"
CICLO = "2025"

In [7]:
# ------------------------------------------------------- 1. arquitetura

arq = pd.read_csv(ARQUITETURA, dtype=str, keep_default_na=False)
col_ciclo = f"original_name_{CICLO}"
for c in ("name", "bigquery_type", col_ciclo):
    assert c in arq.columns, f"arquitetura sem coluna {c}: {list(arq.columns)}"

selected_variables = [n.strip() for n in arq["name"] if n.strip()]
dicionario_colunas = {
    r[col_ciclo].strip(): r["name"].strip()
    for _, r in arq.iterrows()
    if r[col_ciclo].strip() and r["name"].strip()
}
tipos = dict(zip(arq["name"], arq["bigquery_type"], strict=True))

assert len(selected_variables) == len(set(selected_variables)), (
    "name duplicado"
)
assert len(dicionario_colunas) == len(
    [v for v in arq[col_ciclo] if v.strip()]
), f"{col_ciclo} duplicado"

print(
    f"arquitetura: {len(selected_variables)} colunas, "
    f"{len(dicionario_colunas)} mapeadas em {CICLO}"
)

arquitetura: 295 colunas, 206 mapeadas em 2025


In [8]:
import json

_, meta = pyreadstat.read_sav(SAV, metadataonly=True)
vv = meta.variable_value_labels

precisamos = [
    # tarefa 1 — colunas novas de 2025
    "Option_LDW",
    "Option_PQ",
    "Option_ICTQ",
    "Option_FLA",
    "Option_UH",
    "OCOP1",
    "OCOP2",
    "COBN_P1",
    "COBN_P2",
    "P1ISCED",
    "P2ISCED",
    # tarefa 2 — colunas compartilhadas com anos anteriores
    "NatCen",
    "STRATUM",
    "SUBNATIO",
    "OECD",
    "ADMINMODE",
    "LANGTEST_QQQ",
    "LANGTEST_COG",
    "BOOKID",
    "ST001D01T",
    "ST004D01T",
    "EFFORT1",
    "EFFORT2",
    "OCOD3",
    "PROGN",
    "COBN_S",
    "LANGN",
    "ISCEDP",
    "HISCED",
    "IMMIG",
    "REPEAT",
]

filtrado = {k: vv[k] for k in precisamos if k in vv}
faltando = [k for k in precisamos if k not in vv]
print(f"{len(filtrado)}/{len(precisamos)} encontradas no .sav")
if faltando:
    print("AVISO — sem value label no .sav:", faltando)

caminho = (
    BASE + "extra/rotulos_2025.json"
)  # BASE já definido nas celulas anteriores
with open(caminho, "w", encoding="utf-8") as f:
    json.dump(filtrado, f, ensure_ascii=False, indent=2)
print("salvo em", caminho)

31/31 encontradas no .sav
salvo em /content/drive/Shareddrives/Base dos Dados - Geral/Dados/Conjuntos/mundo_oecd_pisa/extra/rotulos_2025.json


In [9]:
# ------------------------------------------------- 2. checagem do .sav

_, meta = pyreadstat.read_sav(SAV, metadataonly=True)
disponiveis = set(meta.column_names)
n_sav = meta.number_rows

usecols = [c for c in dicionario_colunas if c in disponiveis]
ausentes = [c for c in dicionario_colunas if c not in disponiveis]

print(
    f".sav: {n_sav:,} linhas x {len(disponiveis)} colunas | usecols: {len(usecols)}"
)
if ausentes:
    raise SystemExit(
        f"PARE. {len(ausentes)} nomes de {col_ciclo} nao existem no .sav: {ausentes}\n"
        "Corrija a arquitetura antes de continuar."
    )

.sav: 755,721 linhas x 885 colunas | usecols: 206


In [10]:
# ------------------------------------------------------------ 3. leitura

print("lendo o .sav (alguns minutos)...")
df = pd.read_spss(SAV, usecols=usecols, convert_categoricals=False)
assert len(df) == n_sav, f"leitura truncada: {len(df):,} de {n_sav:,}"
print(f"lido: {df.shape[0]:,} x {df.shape[1]}")

lendo o .sav (alguns minutos)...
lido: 755,721 x 206


In [11]:
df = df.rename(columns=dicionario_colunas)
assert not df.columns.duplicated().any(), "colisao de nomes apos rename"

In [12]:
falhas = []
for coluna, tipo in tipos.items():
    if coluna not in df.columns:
        continue
    try:
        if tipo == "int64":
            df[coluna] = df[coluna].astype("Int64")
        elif tipo in ("float", "float64"):
            df[coluna] = df[coluna].astype("float64")
        elif tipo == "date":
            df[coluna] = pd.to_datetime(
                df[coluna].astype("string").str.strip().str[:7],
                format="%d%b%y",
                errors="coerce",
            )
        elif tipo == "string":
            if df[coluna].dtypes == "float64":
                df[coluna] = df[coluna].astype("Int64").astype("string")
            else:
                df[coluna] = df[coluna].astype("string")
    except (ValueError, TypeError) as e:
        falhas.append((coluna, tipo, f"{type(e).__name__}: {e}"))

if falhas:
    print(f"{len(falhas)} colunas nao converteram:")
    for c, t, e in falhas:
        print(f"   {c} -> {t}: {e}")

In [13]:
# ------------------------------------- 6. colunas descontinuadas e ordem

faltantes = [c for c in selected_variables if c not in df.columns]
if faltantes:
    # de uma vez so: 87 insercoes separadas fragmentam o frame e disparam
    # PerformanceWarning a cada chamada
    df = pd.concat(
        [df, pd.DataFrame("", index=df.index, columns=faltantes)],
        axis=1,
        copy=False,
    )
print(f"{len(faltantes)} colunas sem dado em {CICLO}, preenchidas com vazio")

# `year` e particao do caminho (output/year=2025), nao coluna do arquivo
colunas_saida = [c for c in selected_variables if c != "year"]
df = df[colunas_saida].copy()
print(
    f"saida: {df.shape[0]:,} linhas x {df.shape[1]} colunas "
    f"({df.shape[1] - 1} no CSV, sem {PARTICAO})"
)

89 colunas sem dado em 2025, preenchidas com vazio
saida: 755,721 linhas x 294 colunas (293 no CSV, sem country_id_iso_3)


In [14]:
if os.path.isdir(OUTPUT):
    antigos = [
        os.path.join(r, f)
        for r, _, fs in os.walk(OUTPUT)
        for f in fs
        if f == "student.csv"
    ]
    if antigos:
        print(f"removendo {len(antigos)} student.csv de execucao anterior")
    shutil.rmtree(OUTPUT)
Path(OUTPUT).mkdir(parents=True, exist_ok=True)

In [15]:
print(f"particionando por {PARTICAO}...")
to_partitions(data=df, partition_columns=[PARTICAO], savepath=OUTPUT)

particionando por country_id_iso_3...


In [ ]:
# --------------------------------------------------------- 9. conferencia

arquivos = [
    os.path.join(r, f)
    for r, _, fs in os.walk(OUTPUT)
    for f in fs
    if f == "student.csv"
]
total = sum(contar_linhas(p) for p in arquivos)

print(
    f"\n{len(arquivos)} particoes | linhas nos CSVs: {total:,} | no .sav: {n_sav:,}"
)

amostra = pd.read_csv(arquivos[0], dtype=str, keep_default_na=False, nrows=3)
pais = Path(arquivos[0]).parent.name
print(f"amostra {pais}: {amostra.shape[1]} colunas")
print(f"  {PARTICAO} fora do CSV: {PARTICAO not in amostra.columns}")
print(f"  year fora do CSV:            {'year' not in amostra.columns}")

ok = (
    total == n_sav
    and amostra.shape[1] == len(colunas_saida) - 1
    and PARTICAO not in amostra.columns
    and "year" not in amostra.columns
)
print("\nOK" if ok else "\nErro")

In [ ]:
arquivos = sorted(glob.glob(f"{OUTPUT}/country_id_iso_3=*/student.csv"))
print(len(arquivos), "partições")

p = next(x for x in arquivos if "BRA" in x)
d = pd.read_csv(p, dtype=str, keep_default_na=False)

In [ ]:
q = """
SELECT *
FROM `basedosdados.world_oecd_pisa.student`
LIMIT 0
"""
bq = bd.read_sql(q, billing_project_id="basedosdados-dev")
print(bq)

novas = [c for c in d.columns if c not in bq]
sumiram = [
    c
    for c in bq
    if c not in d.columns and c not in ("year", "country_id_iso_3")
]
print(f"{len(novas)} colunas novas:")
for c in novas:
    print("  ", c)
print(f"\n{len(sumiram)} colunas do BQ ausentes no CSV:", sumiram or "nenhuma")

## Baixar anos anteriores

In [ ]:
cli = bigquery.Client(project="basedosdados-dev")
cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

sql = """SELECT * EXCEPT(year, country_id_iso_3) FROM `basedosdados.world_oecd_pisa.student`
          WHERE year = 2025 AND country_id_iso_3 = 'BRA'"""
job = cli.query(sql, job_config=cfg)
gb = job.total_bytes_processed / 1e9
print(
    f"{gb:.2f} GB por consulta -> ~{gb * 300:.0f} GB nas 300 -> ~US$ {gb * 300 / 1000 * 6.25:.2f}"
)

In [ ]:
for ano in (2015, 2018, 2022, 2025):
    sql = f"""SELECT * EXCEPT(year, country_id_iso_3) FROM `basedosdados.world_oecd_pisa.student`
              WHERE year = {ano} AND country_id_iso_3 = 'BRA'"""
    gb = cli.query(sql, job_config=cfg).total_bytes_processed / 1e9
    print(f"{ano}: {gb:.3f} GB")

# quanto pesa a tabela inteira, como referencia
total = (
    cli.query(
        "SELECT * FROM `basedosdados.world_oecd_pisa.student`", job_config=cfg
    ).total_bytes_processed
    / 1e9
)
print(f"tabela inteira: {total:.2f} GB")

In [ ]:
# ----------------------------------------------------------------- config

PROJETO = "basedosdados-dev"  # billing project no GCP
TABELA = "basedosdados.world_oecd_pisa.student"

# Grava local primeiro: o FUSE do Drive e lento para ~300 arquivos pequenos e
# ocasionalmente falha em silencio. A copia para o Drive acontece no fim.
TRABALHO = Path("/content/download_student")
DRIVE = Path(BASE) / "output/novas_colunas"

ANOS = None  # None = todos; ou [2015, 2018] para restringir
TENTATIVAS = 3  # por particao, em caso de falha de rede

In [ ]:
# ------------------------------------------------------ 1. lista de particoes
# Traz tambem a contagem de linhas, que serve para validar cada download.

sql_pares = f"""
    SELECT year, country_id_iso_3, COUNT(*) AS linhas
    FROM `{TABELA}`
    {"WHERE year IN (" + ",".join(map(str, ANOS)) + ")" if ANOS else ""}
    GROUP BY 1, 2
    ORDER BY 1, 2
"""
pares = bd.read_sql(sql_pares, billing_project_id=PROJETO)
print(f"{len(pares)} particoes, {pares['linhas'].sum():,} linhas no total")
print(pares.groupby("year")["linhas"].agg(["count", "sum"]).to_string())

In [ ]:
# ---------------------------------------------------------------- 2. download

pendentes, divergentes = [], []
t0 = time.time()

for k, (_, r) in enumerate(pares.iterrows(), 1):
    pasta = (
        TRABALHO / f"year={r.year}" / f"country_id_iso_3={r.country_id_iso_3}"
    )
    arquivo = pasta / "student.csv"
    rotulo = f"{r.year} {r.country_id_iso_3}"

    if arquivo.exists():
        continue

    sql = f"""
        SELECT * EXCEPT(year, country_id_iso_3)
        FROM `{TABELA}`
        WHERE year = {r.year} AND country_id_iso_3 = '{r.country_id_iso_3}'
    """

    d = None
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            d = bd.read_sql(sql, billing_project_id=PROJETO)
            break
        except Exception as e:
            print(
                f"  [{k}/{len(pares)}] {rotulo}: tentativa {tentativa} falhou "
                f"({type(e).__name__})"
            )
            if tentativa < TENTATIVAS:
                time.sleep(5 * tentativa)

    if d is None:
        pendentes.append(rotulo)
        continue

    # Nao aborta o laco: registra a divergencia e segue, para nao perder as
    # particoes restantes por causa de uma. Revisar a lista no fim.
    if len(d) != r.linhas:
        divergentes.append((rotulo, len(d), int(r.linhas)))
        print(
            f"  [{k}/{len(pares)}] {rotulo}: DIVERGENCIA {len(d)} != {r.linhas}"
        )

    pasta.mkdir(parents=True, exist_ok=True)
    d.to_csv(arquivo, index=False)
    print(f"[{k}/{len(pares)}] {rotulo} {len(d):>7,} linhas")
    del d

print(f"\ntempo: {(time.time() - t0) / 60:.1f} min")

In [ ]:
import shutil
from pathlib import Path

TRABALHO = Path("/content/download_student")
DRIVE = Path(BASE) / "output/novas_colunas"

baixados = sorted(TRABALHO.glob("year=*/country_id_iso_3=*/student.csv"))
print(f"{len(baixados)} arquivos em {TRABALHO}")
print("esperado: 233")

if len(baixados) == 233:
    DRIVE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(TRABALHO, DRIVE, dirs_exist_ok=True)
    n = len(list(DRIVE.glob("year=*/country_id_iso_3=*/student.csv")))
    print(f"{n} arquivos no Drive")
else:
    print("faltou arquivo -- nao copie ainda")

### Adicionando as 45 colunas novas na ordem de 2025

In [ ]:
# ----------------------------------------------------------------- config

DRIVE = Path(BASE) / "output/novas_colunas"
TRABALHO = Path("/content/uniformizado")
REFERENCIA = "year=2025"  # de onde sai a ordem canonica das colunas

# ------------------------------------------------------- 1. ordem canonica

algum_2025 = next(
    iter(sorted((DRIVE / REFERENCIA).glob("*/student.csv"))), None
)
if algum_2025 is None:
    raise SystemExit(f"Nenhum arquivo em {DRIVE / REFERENCIA}")

ORDEM = pd.read_csv(algum_2025, nrows=0).columns.tolist()
print(f"ordem de referencia: {len(ORDEM)} colunas ({algum_2025.parts[-3]})")

In [ ]:
# --------------------------------------------------- 2. inventario de entrada

from collections import Counter

arquivos = sorted(DRIVE.glob("year=*/country_id_iso_3=*/student.csv"))
print(f"{len(arquivos)} arquivos encontrados")

antes = {}
for p in arquivos:
    cols = pd.read_csv(p, nrows=0).columns.tolist()
    sobra = [c for c in cols if c not in ORDEM]
    if sobra:
        raise SystemExit(
            f"PARE. {p.parts[-3]}/{p.parts[-2]} tem colunas fora da referencia: {sobra}\n"
            "Investigue antes de continuar: reordenar descartaria esses dados."
        )
    antes[p] = len(cols)

print("colunas por arquivo:", dict(Counter(antes.values())))

In [ ]:
if TRABALHO.exists():
    shutil.rmtree(TRABALHO)

feitos, ja_ok = 0, 0
for p in arquivos:
    ano, pais = p.parts[-3], p.parts[-2]

    d = pd.read_csv(p, dtype=str, keep_default_na=False)
    n_linhas = len(d)

    faltantes = [c for c in ORDEM if c not in d.columns]
    if faltantes:
        d = pd.concat(
            [d, pd.DataFrame("", index=d.index, columns=faltantes)],
            axis=1,
            copy=False,
        )
    else:
        ja_ok += 1

    d = d[ORDEM].copy()

    assert len(d) == n_linhas, f"{ano}/{pais}: perdeu linha"
    assert list(d.columns) == ORDEM, f"{ano}/{pais}: ordem errada"

    destino = TRABALHO / ano / pais
    destino.mkdir(parents=True, exist_ok=True)
    d.to_csv(destino / "student.csv", index=False)

    feitos += 1
    if feitos % 25 == 0 or faltantes:
        print(
            f"[{feitos}/{len(arquivos)}] {ano} {pais}  "
            f"{n_linhas:>7,} linhas  +{len(faltantes)} colunas"
        )
    del d

print(f"\n{feitos} arquivos gravados em {TRABALHO}")
print(f"{ja_ok} ja estavam completos (nenhuma coluna acrescentada)")

In [ ]:
from pathlib import Path

import pandas as pd

DRIVE = Path(BASE) / "output/novas_colunas"
TRABALHO = Path("/content/uniformizado")

ORDEM = pd.read_csv(
    next(iter(sorted((TRABALHO / "year=2025").glob("*/student.csv")))), nrows=0
).columns.tolist()
print(f"{len(ORDEM)} colunas na referencia")

problemas = []
for p in sorted(DRIVE.glob("year=*/country_id_iso_3=*/student.csv")):
    novo = TRABALHO / p.parts[-3] / p.parts[-2] / "student.csv"
    if not novo.exists():
        problemas.append((p.parts[-3], p.parts[-2], "nao gerado"))
        continue
    if pd.read_csv(novo, nrows=0).columns.tolist() != ORDEM:
        problemas.append((p.parts[-3], p.parts[-2], "ordem divergente"))
        continue
    a = contar_linhas(p)
    b = contar_linhas(novo)
    if a != b:
        problemas.append((p.parts[-3], p.parts[-2], f"{b} != {a}"))

print(f"{len(problemas)} problemas", problemas[:10])

In [ ]:
import shutil
from pathlib import Path

TRABALHO = Path("/content/uniformizado")
DRIVE = Path(BASE) / "output/novas_colunas"

shutil.copytree(TRABALHO, DRIVE, dirs_exist_ok=True)
print(
    len(list(DRIVE.glob("year=*/country_id_iso_3=*/student.csv"))),
    "arquivos no Drive",
)

In [ ]:
import pandas as pd

d = pd.read_csv(
    DRIVE / "year=2015/country_id_iso_3=BRA/student.csv",
    dtype=str,
    keep_default_na=False,
)
novas = [
    "plausible_value_1_ldw",
    "option_learning_digital_world",
    "isced_parent_1",
    "wle_ai_use_school",
]
print(d.shape)
print({c: (d[c] == "").all() for c in novas})  # todos devem dar True

In [ ]:
import os
import shutil
from pathlib import Path

ORIGEM = Path("/content/download_student")
DESTINO = Path(BASE) / "backup"

print("origem existe:", ORIGEM.exists())
if ORIGEM.exists():
    n = len(list(ORIGEM.glob("year=*/country_id_iso_3=*/student.csv")))
    print(f"{n} arquivos (esperado 233)")

# compacta local
zip_local = shutil.make_archive("/content/download_student_bq", "zip", ORIGEM)
mb = os.path.getsize(zip_local) / 1e6
print(f"{zip_local}  {mb:.0f} MB")

DESTINO.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_local, DESTINO / "download_student_bq.zip")

final = DESTINO / "download_student_bq.zip"
print(f"no Drive: {final}  {os.path.getsize(final) / 1e6:.0f} MB")

### Comparar com output2 (validação)

In [16]:
# ---- comparar BRA: output novo x output2 (script antigo) ----
# ajuste os dois caminhos abaixo se as pastas tiverem outro nome no seu Drive

PASTA_NOVA = Path(BASE) / "output" / "novas_colunas"
PASTA_ANTIGA = Path(BASE) / "output2"

anos_nova = {p.name.split("=")[1] for p in PASTA_NOVA.glob("year=*")}
anos_antiga = {p.name.split("=")[1] for p in PASTA_ANTIGA.glob("year=*")}
anos = sorted(anos_nova & anos_antiga, key=int)
print(f"anos em comum: {anos}")

# colunas que sabidamente divergem por design (fix aplicado so no dbt, nao no
# CSV bruto) -- reportadas a parte, sem contar como divergencia real
CONHECIDAS = {
    "country_id_m49"
}  # zero a esquerda: fix e so no safe_cast do dbt

for ano in anos:
    a = PASTA_NOVA / f"year={ano}" / "country_id_iso_3=BRA" / "student.csv"
    b = PASTA_ANTIGA / f"year={ano}" / "country_id_iso_3=BRA" / "student.csv"
    if not a.exists() or not b.exists():
        print(
            f"{ano}: arquivo faltando ({'nova' if not a.exists() else 'antiga'})"
        )
        continue

    da = pd.read_csv(a, dtype=str, keep_default_na=False)
    db = pd.read_csv(b, dtype=str, keep_default_na=False)

    so_nova = [c for c in da.columns if c not in db.columns]
    so_antiga = [c for c in db.columns if c not in da.columns]
    comuns = [c for c in da.columns if c in db.columns]

    print(
        f"\n{ano}: {da.shape[0]} x {da.shape[1]} (nova) vs {db.shape[0]} x {db.shape[1]} (antiga)"
    )
    if so_nova:
        print("  colunas so na nova:", so_nova)
    if so_antiga:
        print("  colunas so na antiga:", so_antiga)

    if da.shape[0] != db.shape[0]:
        print("  numero de linhas diferente -- pulando comparacao de valores")
        continue

    chave = "student_id" if "student_id" in comuns else comuns
    da_cmp = da[comuns].sort_values(chave).reset_index(drop=True)
    db_cmp = db[comuns].sort_values(chave).reset_index(drop=True)

    dif = da_cmp.ne(db_cmp)
    colunas_dif = list(dif.any(axis=0)[dif.any(axis=0)].index)
    esperadas = [c for c in colunas_dif if c in CONHECIDAS]
    inesperadas = [c for c in colunas_dif if c not in CONHECIDAS]

    if not colunas_dif:
        print("  dados identicos")
    else:
        if esperadas:
            print("  divergem (esperado, fix so no dbt):", esperadas)
        if inesperadas:
            linhas_dif = dif[inesperadas].any(axis=1).sum()
            print(
                f"  divergem (NAO esperado): {inesperadas} -- {linhas_dif} linhas afetadas"
            )

anos em comum: ['2025']

2025: 30140 x 293 (nova) vs 30140 x 293 (antiga)
  colunas so na nova: ['wle_ict_availability_school']
  colunas so na antiga: ['wle_availability_usage_school']
  dados identicos
